<a href="https://colab.research.google.com/github/ysuter/FHNW-BAI-ComputerVision/blob/main/FFT_Intuition_Tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fast Fourier Transform (FFT) für Bildverarbeitung
## Intuition für räumliche Frequenzen entwickeln

---

**Willkommen!** Dieses Notebook hilft Ihnen, ein intuitives Verständnis der Fast Fourier Transform zu entwickeln und zu verstehen, wie sie verborgene Muster in Bildern aufdeckt.

### 🎯 Lernziele:
- ✅ Verstehen, was **räumliche Frequenzen** in Bildern bedeuten
- ✅ **Fourier-Spektren** visualisieren und interpretieren
- ✅ Sehen, wie Bilder aus **verschiedenen Frequenzen zusammengesetzt** sind
- ✅ FFT für praktische Bildverarbeitungsaufgaben anwenden
- ✅ **Intuition** durch interaktives Experimentieren aufbauen

### 🚫 Out of Scope:
- Komplexe mathematische Herleitungen
- Rigorose Beweise
- Signalverarbeitungstheorie

### ✅ Worauf wir fokussieren:
- Visuelles Verständnis
- Praktisches Experimentieren
- Anwendungen
- Intuition aufbauen!

---

## 📦 Setup & Installation

In [ ]:
# Bibliotheken installieren
!pip install opencv-python-headless matplotlib numpy scipy ipywidgets scikit-image -q

# Imports
import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from scipy import ndimage
from skimage import data
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from google.colab import files

# Matplotlib konfigurieren
plt.rcParams['figure.figsize'] = (16, 8)
plt.rcParams['font.size'] = 11
%matplotlib inline

print("✅ Alle Bibliotheken erfolgreich geladen!")
print(f"NumPy Version: {np.__version__}")
print(f"OpenCV Version: {cv2.__version__}")

---

# 🎨 Teil 1: Intuition aufbauen - Was SIND räumliche Frequenzen?

Bevor wir in FFT eintauchen, bauen wir Intuition dafür auf, was "Frequenzen" in Bildern bedeuten.

## 🤔 Die zentrale Frage:

> **"Wenn ein Bild nur Pixel mit Helligkeitswerten ist, was bedeutet dann 'Frequenz'?"**

---

## 1.1 Analogie: Audio-Frequenzen

**Bei Audio:**
- Niedrige Frequenzen = Bass (langsame Schwingungen, wie 🎸)
- Hohe Frequenzen = Höhen (schnelle Schwingungen, wie 🎺)
- Ein Lied ist eine **Mischung** vieler Frequenzen

**Bei Bildern:**
- Niedrige Frequenzen = Glatte Bereiche, graduelle Änderungen (wie ein Sonnenuntergangs-Verlauf)
- Hohe Frequenzen = Kanten, Details, Texturen (wie Haarsträhnen)
- Ein Bild ist eine **Mischung** vieler räumlicher Frequenzen

### 💡 Kernaussage:
**Frequenz in Bildern = Wie schnell sich Pixelwerte ändern**
- Glatter Verlauf → Niedrige Frequenz
- Scharfe Kante → Hohe Frequenz

## 1.2 Visuelle Demonstration: Reine Frequenzen

Erstellen wir Bilder mit **nur** spezifischen Frequenzen, um zu sehen wie sie aussehen!

In [ ]:
def create_sinusoidal_pattern(size=256, frequency=5, angle=0):
    """
    Erstellt ein sinusförmiges Muster - ein Bild mit 'reiner Frequenz'

    Parameter:
    - size: Bildgröße (size x size)
    - frequency: Wie viele Zyklen über das Bild
    - angle: Rotationswinkel in Grad
    """
    # Koordinatengitter erstellen
    x = np.linspace(-np.pi, np.pi, size)
    y = np.linspace(-np.pi, np.pi, size)
    X, Y = np.meshgrid(x, y)

    # Koordinaten rotieren
    angle_rad = np.deg2rad(angle)
    X_rot = X * np.cos(angle_rad) - Y * np.sin(angle_rad)

    # Sinusförmiges Muster erstellen
    pattern = np.sin(frequency * X_rot)

    # Auf 0-255 normalisieren
    pattern = ((pattern + 1) * 127.5).astype(np.uint8)

    return pattern

# Muster mit verschiedenen Frequenzen erstellen
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

frequencies = [1, 3, 8, 20]
for i, freq in enumerate(frequencies):
    pattern = create_sinusoidal_pattern(frequency=freq)
    axes[i].imshow(pattern, cmap='gray')
    axes[i].set_title(f'Frequenz = {freq}\n({"NIEDRIG" if freq <= 3 else "MITTEL" if freq <= 10 else "HOCH"})',
                     fontsize=13, fontweight='bold')
    axes[i].axis('off')

plt.suptitle('Reine räumliche Frequenzen - Sehen Sie das Muster?', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 Beobachtung:")
print("   • NIEDRIGE Frequenz (1): Glatt, graduelle Änderungen - nur wenige Streifen")
print("   • MITTLERE Frequenz (3, 8): Mehr Streifen, schnellere Änderungen")
print("   • HOHE Frequenz (20): Viele dünne Streifen, rapide Änderungen")
print("\n→ Echte Bilder sind Kombinationen ALLER dieser Frequenzen!")

### 🎮 Interaktiv: Erstelle ein eigenes Muster!

In [ ]:
# Interaktiver Mustergenerator
freq_slider = widgets.IntSlider(
    value=5, min=1, max=30, step=1,
    description='Frequenz:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='500px')
)

angle_slider = widgets.IntSlider(
    value=0, min=0, max=180, step=15,
    description='Winkel (°):',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='500px')
)

output = widgets.Output()

def update_pattern(change):
    with output:
        clear_output(wait=True)
        pattern = create_sinusoidal_pattern(
            frequency=freq_slider.value,
            angle=angle_slider.value
        )

        plt.figure(figsize=(8, 8))
        plt.imshow(pattern, cmap='gray')
        plt.title(f'Räumliche Frequenz = {freq_slider.value}, Winkel = {angle_slider.value}°',
                 fontsize=14, fontweight='bold')
        plt.axis('off')
        plt.show()

        category = "NIEDRIG" if freq_slider.value <= 5 else "MITTEL" if freq_slider.value <= 15 else "HOCH"
        print(f"\n📊 Kategorie: {category}e Frequenz")
        print(f"   Streifen über das Bild: ~{freq_slider.value * 2}")

freq_slider.observe(update_pattern, names='value')
angle_slider.observe(update_pattern, names='value')

display(freq_slider, angle_slider, output)
update_pattern(None)

print("\n🎯 Probieren Sie:")
print("   1. Bewegen Sie Frequenz von niedrig zu hoch - sehen Sie wie Streifen dünner werden?")
print("   2. Ändern Sie den Winkel - die Frequenzrichtung ist wichtig!")
print("   3. In solche Komponenten zerlegt FFT Bilder!")

---

# 🔍 Teil 2: Die Fourier-Transformation - Bilder zerlegen

## 2.1 Die große Idee

### 🎼 Die musikalische Analogie:

Stellen dir vor, du hast eine Aufnahme eines Orchesters:
- **Zeitbereich**: Du hören alle Instrumente gemischt
- **Frequenzbereich** (nach Fourier-Transform):Du kannst sehen welche Töne aktuell zu hören sind. Zudem haben die Instrumente ein charakteristisches Spektrum

### 📸 Für Bilder:

- **Ortsbereich** (normales Bild): Du siehst Pixel
- **Frequenzbereich** (nach FFT): Du siehts, sehen welche Frequenzen das Bild bilden

---

## 2.2 Was macht FFT eigentlich?

**FFT beantwortet die Frage:**

> *"Wie viel von jeder Frequenz ist in meinem Bild vorhanden, und in welche Richtung?"*

**Ausgabe (Frequenzspektrum):**
- **Zentrum**: Niedrige Frequenzen (DC-Komponente, mittlere Helligkeit)
- **Ränder**: Hohe Frequenzen (Details, Kanten)
- **Helligkeit**: Wie viel von dieser Frequenz existiert
- **Position**: Richtung dieser Frequenz

---

## 2.3 Hilfsfunktionen für FFT-Visualisierung

In [ ]:
def compute_fft(image):
    """
    FFT berechnen und Nullfrequenz in die Mitte verschieben
    """
    # In Graustufen konvertieren falls nötig
    if len(image.shape) == 3:
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    else:
        gray = image.astype(float)

    # FFT berechnen
    f_transform = np.fft.fft2(gray)
    f_shift = np.fft.fftshift(f_transform)  # Nullfrequenz in Mitte verschieben

    return f_shift, gray

def visualize_fft_spectrum(f_shift, title="Fourier-Spektrum", log_scale=True):
    """
    Magnitude-Spektrum visualisieren
    """
    # Magnitude berechnen
    magnitude = np.abs(f_shift)

    if log_scale:
        # Log-Skala für bessere Visualisierung
        magnitude_display = np.log(magnitude + 1)
    else:
        magnitude_display = magnitude

    # Plotten
    plt.figure(figsize=(10, 8))
    plt.imshow(magnitude_display, cmap='hot')
    plt.colorbar(label='Magnitude (Log-Skala)' if log_scale else 'Magnitude', fraction=0.046)
    plt.title(title, fontsize=14, fontweight='bold')
    plt.axis('off')

    # Zentrumsmarkierung hinzufügen
    center_y, center_x = magnitude.shape[0] // 2, magnitude.shape[1] // 2
    plt.plot(center_x, center_y, 'b+', markersize=15, markeredgewidth=2)
    plt.text(center_x + 10, center_y, 'DC\n(Niedrige Freq)', color='cyan', fontsize=10, fontweight='bold')

    plt.tight_layout()
    plt.show()

    return magnitude

def show_fft_analysis(image, title="FFT-Analyse"):
    """
    Vollständige FFT-Analyse mit Original, Spektrum und Phase
    """
    f_shift, gray = compute_fft(image)
    magnitude = np.abs(f_shift)
    phase = np.angle(f_shift)

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    # Originalbild
    axes[0].imshow(gray, cmap='gray')
    axes[0].set_title('Originalbild', fontsize=13, fontweight='bold')
    axes[0].axis('off')

    # Magnitude-Spektrum (Log-Skala)
    magnitude_log = np.log(magnitude + 1)
    im1 = axes[1].imshow(magnitude_log, cmap='hot')
    axes[1].set_title('Magnitude-Spektrum\n(Welche Frequenzen existieren)', fontsize=13, fontweight='bold')
    axes[1].axis('off')
    plt.colorbar(im1, ax=axes[1], fraction=0.046)

    # Phasen-Spektrum
    im2 = axes[2].imshow(phase, cmap='twilight')
    axes[2].set_title('Phasen-Spektrum\n(Wo Frequenzen lokalisiert sind)', fontsize=13, fontweight='bold')
    axes[2].axis('off')
    plt.colorbar(im2, ax=axes[2], fraction=0.046)

    plt.suptitle(title, fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

    return f_shift, magnitude, phase

print("✅ FFT-Hilfsfunktionen definiert!")

---

# 🧪 Teil 3: Experimente - Intuition aufbauen

Analysieren wir jetzt verschiedene Bildtypen, um zu verstehen was ihre FFT offenbart!

## 3.1 Experiment 1: Reine Frequenzen

**Frage:** Wie sieht die FFT unserer sinusförmigen Muster aus?

In [ ]:
# Reine Frequenzmuster erstellen
pattern_low = create_sinusoidal_pattern(frequency=3, angle=0)
pattern_high = create_sinusoidal_pattern(frequency=15, angle=45)

print("🔬 Analysiere NIEDRIGES Frequenzmuster:")
show_fft_analysis(pattern_low, "FFT von niedrigem Frequenzmuster")

print("\n" + "="*80 + "\n")
print("🔬 Analysiere HOHES Frequenzmuster:")
show_fft_analysis(pattern_high, "FFT von hohem Frequenzmuster")

print("\n💡 Wichtige Beobachtungen:")
print("   1. Reine Frequenzen erscheinen als HELLE PUNKTE im Spektrum")
print("   2. Niedrige Frequenz → Punkte nahe am Zentrum")
print("   3. Hohe Frequenz → Punkte weit vom Zentrum entfernt")
print("   4. Winkel des Musters → Winkel der Punkte im Spektrum")
print("   5. Immer symmetrisch (zwei Punkte) aufgrund mathematischer Eigenschaften")

## 3.2 Experiment 2: Einfache geometrische Formen

**Frage:** Aus welchen Frequenzen bestehen einfache Formen?

In [ ]:
# Einfache Testbilder erstellen
def create_test_images():
    size = 256

    # 1. Einzelne vertikale Kante
    edge = np.zeros((size, size), dtype=np.uint8)
    edge[:, :size//2] = 255

    # 2. Schachbrett
    checker = np.zeros((size, size), dtype=np.uint8)
    block_size = 32
    for i in range(0, size, block_size):
        for j in range(0, size, block_size):
            if (i // block_size + j // block_size) % 2 == 0:
                checker[i:i+block_size, j:j+block_size] = 255

    # 3. Glatter Verlauf
    gradient = np.linspace(0, 255, size, dtype=np.uint8)
    gradient = np.tile(gradient, (size, 1))

    # 4. Kreis
    circle = np.zeros((size, size), dtype=np.uint8)
    cv2.circle(circle, (size//2, size//2), size//4, 255, -1)

    return edge, checker, gradient, circle

edge, checker, gradient, circle = create_test_images()

# Jedes analysieren
test_images = [
    (edge, "Scharfe Kante", "Viele HOHE Frequenzen (scharfer Übergang)"),
    (checker, "Schachbrett", "Reguläres Muster = spezifische Frequenzen"),
    (gradient, "Glatter Verlauf", "Nur NIEDRIGE Frequenzen (sanfte Änderung)"),
    (circle, "Kreis", "Mix von Frequenzen für gekrümmte Kanten")
]

for img, title, description in test_images:
    print(f"\n{'='*80}")
    print(f"🔬 {title}: {description}")
    show_fft_analysis(img, f"FFT-Analyse: {title}")
    print()

### 📝 Interpretationshilfe:

| Bildtyp | Spektrum-Erscheinung | Warum? |
|---------|---------------------|--------|
| **Scharfe Kante** | Helle horizontale Linie durch Zentrum | Vertikale Kante = horizontale Frequenzen |
| **Schachbrett** | Punkte an Ecken | Reguläres Muster = spezifische Frequenzen |
| **Glatter Verlauf** | Nur Zentrum sehr hell | Nur niedrige Frequenzen (graduelle Änderung) |
| **Kreis** | Kreisförmiges Muster im Spektrum | Gekrümmte Kanten = Frequenzen in alle Richtungen |

## 3.3 Experiment 3: Echtes Bild analysieren

Analysieren wir jetzt eine **echte Fotografie**, um alle zusammen gemischten Frequenzen zu sehen!

In [ ]:
# Beispielbild von scikit-image laden
# Du kannst auch ein eigenes hochladen!

# Option 1: Eingebautes Bild verwenden
real_image = data.camera()  # Klassisches Testbild

print("📸 Analysiere ECHTE Fotografie:")
print("   Dieses Bild enthält ALLE Frequenzen gemischt!\n")

f_shift, magnitude, phase = show_fft_analysis(real_image, "FFT einer echten Fotografie")

print("\n💡 Was wir sehen:")
print("   ✓ Helles ZENTRUM: Niedrige Frequenzen (Gesamtform, glatte Bereiche)")
print("   ✓ Dunklere Ränder: Hohe Frequenzen (Details, Texturen, Kanten)")
print("   ✓ Kreuzmuster: Dominante horizontale & vertikale Features")
print("   ✓ Keine einzelnen Punkte: Echte Bilder sind KOMPLEXE Mischungen!")

### 🎮 Interaktiv: Analysieren dein eigenes Bild!

In [ ]:
print("📤 Lade ein eigenes Bild hoch, um seinen Frequenzinhalt zu analysieren!")
print("   (Fotos funktionieren am besten - probiere verschiedene Motive)\n")

uploaded = files.upload()

if uploaded:
    filename = list(uploaded.keys())[0]
    user_image = cv2.imread(filename, cv2.IMREAD_GRAYSCALE)

    if user_image is not None:
        # Verkleinern falls zu groß
        if user_image.shape[0] > 512 or user_image.shape[1] > 512:
            scale = 512 / max(user_image.shape)
            user_image = cv2.resize(user_image, None, fx=scale, fy=scale)
            print(f"✓ Auf {user_image.shape} verkleinert für schnellere Verarbeitung\n")

        show_fft_analysis(user_image, f"FFT-Analyse: {filename}")

        print("\n🔍 Achten Sie auf:")
        print("   • Ist das Zentrum sehr hell? → Glatte Bereiche dominieren")
        print("   • Sind die Ränder hell? → Viele Details/Texturen")
        print("   • Sehen Sie Richtungslinien? → Starke Kanten in dieser Richtung")
    else:
        print("❌ Bild konnte nicht geladen werden")
else:
    print("⚠️ Keine Datei hochgeladen")

---

# 🔧 Teil 4: Frequenzfilterung - Praktische Anwendungen

Jetzt wo wir FFT verstehen, nutzen wir es um Bilder zu manipulieren!

## 4.1 Die Macht der Frequenzbereichs-Filterung

**Kernidee:**
1. Bild in Frequenzbereich transformieren (FFT)
2. Bestimmte Frequenzen modifizieren
3. Zurücktransformieren (Inverse FFT)

**Warum ist das mächtig?**
- Rauschen entfernen (meist hohe Frequenz)
- Schärfen (hohe Frequenzen verstärken)
- Kompression (Frequenzen verwerfen, die Menschen nicht bemerken)

---

## 4.2 Frequenzfilter erstellen

In [ ]:
def create_circular_mask(shape, radius, filter_type='lowpass'):
    """
    Kreisförmige Maske für Frequenzfilterung erstellen

    Parameter:
    - shape: Bildform (Zeilen, Spalten)
    - radius: Radius des Filters
    - filter_type: 'lowpass' oder 'highpass'
    """
    rows, cols = shape
    crow, ccol = rows // 2, cols // 2

    # Koordinatengitter erstellen
    y, x = np.ogrid[:rows, :cols]

    # Distanz vom Zentrum berechnen
    distance = np.sqrt((x - ccol)**2 + (y - crow)**2)

    # Maske erstellen
    if filter_type == 'lowpass':
        mask = (distance <= radius).astype(float)
    else:  # highpass
        mask = (distance > radius).astype(float)

    return mask

def apply_frequency_filter(image, radius, filter_type='lowpass'):
    """
    Frequenzbereichsfilter anwenden
    """
    # FFT berechnen
    f_shift, gray = compute_fft(image)

    # Maske erstellen und anwenden
    mask = create_circular_mask(gray.shape, radius, filter_type)
    f_shift_filtered = f_shift * mask

    # Inverse FFT
    f_ishift = np.fft.ifftshift(f_shift_filtered)
    img_back = np.fft.ifft2(f_ishift)
    img_back = np.abs(img_back)

    return img_back, f_shift, f_shift_filtered, mask, gray

def visualize_filtering_pipeline(image, radius, filter_type='lowpass'):
    """
    Vollständige Filter-Pipeline zeigen
    """
    result, f_orig, f_filtered, mask, gray = apply_frequency_filter(
        image, radius, filter_type
    )

    # Magnitude-Spektren berechnen
    mag_orig = np.log(np.abs(f_orig) + 1)
    mag_filtered = np.log(np.abs(f_filtered) + 1)

    # Visualisierung
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))

    # Zeile 1: Original-Verarbeitung
    axes[0, 0].imshow(gray, cmap='gray')
    axes[0, 0].set_title('1. Originalbild', fontsize=12, fontweight='bold')
    axes[0, 0].axis('off')

    axes[0, 1].imshow(mag_orig, cmap='hot')
    axes[0, 1].set_title('2. FFT-Spektrum', fontsize=12, fontweight='bold')
    axes[0, 1].axis('off')

    axes[0, 2].imshow(mask, cmap='gray')
    axes[0, 2].set_title(f'3. {filter_type.upper()}-Maske\n(Radius={radius})',
                        fontsize=12, fontweight='bold')
    axes[0, 2].axis('off')

    # Zeile 2: Gefilterte Verarbeitung
    axes[1, 0].imshow(mag_filtered, cmap='hot')
    axes[1, 0].set_title('4. Gefiltertes Spektrum', fontsize=12, fontweight='bold')
    axes[1, 0].axis('off')

    axes[1, 1].text(0.5, 0.5, '↓\nInverse FFT\n(IFFT)',
                   ha='center', va='center', fontsize=20, fontweight='bold')
    axes[1, 1].axis('off')

    axes[1, 2].imshow(result, cmap='gray')
    axes[1, 2].set_title('5. Ergebnis', fontsize=12, fontweight='bold')
    axes[1, 2].axis('off')

    filter_name = "TIEFPASS" if filter_type == 'lowpass' else "HOCHPASS"
    plt.suptitle(f'{filter_name}-Filter Pipeline (Radius={radius})',
                fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # Interpretation
    if filter_type == 'lowpass':
        print("\n🔽 TIEFPASS-FILTER Effekt:")
        print("   ✓ Behält niedrige Frequenzen (glatte Bereiche, Struktur)")
        print("   ✓ Entfernt hohe Frequenzen (Details, Kanten, Rauschen)")
        print("   → Ergebnis: VERSCHWOMMENES/GEGLÄTTETES Bild")
        print("\n   💼 Anwendungsfälle: Rauschreduktion, Glättung, Kompression")
    else:
        print("\n🔼 HOCHPASS-FILTER Effekt:")
        print("   ✓ Entfernt niedrige Frequenzen (glatte Bereiche, Struktur)")
        print("   ✓ Behält hohe Frequenzen (Details, Kanten)")
        print("   → Ergebnis: KANTEN/DETAILS betont")
        print("\n   💼 Anwendungsfälle: Kantenerkennung, Schärfung, Feature-Extraktion")

    return result

print("✅ Filterfunktionen bereit!")

## 4.3 Tiefpass-Filter Demo

In [ ]:
# Kamerabild verwenden
test_img = data.camera()

print("🔽 Wende TIEFPASS-Filter an (entfernt hohe Frequenzen)\n")
result_low = visualize_filtering_pipeline(test_img, radius=50, filter_type='lowpass')

## 4.4 Hochpass-Filter Demo

In [ ]:
print("🔼 Wende HOCHPASS-Filter an (entfernt niedrige Frequenzen)\n")
result_high = visualize_filtering_pipeline(test_img, radius=30, filter_type='highpass')

## 4.5 Interaktiver Filter-Explorer

Passe die Filterparameter an und sieh die Effekte in Echtzeit.

In [ ]:
# Interaktive Filtersteuerung
filter_type_dropdown = widgets.Dropdown(
    options=['lowpass', 'highpass'],
    value='lowpass',
    description='Filtertyp:',
    style={'description_width': '100px'}
)

radius_slider = widgets.IntSlider(
    value=50, min=10, max=150, step=10,
    description='Radius:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='500px')
)

output_filter = widgets.Output()

def update_filter(change):
    with output_filter:
        clear_output(wait=True)

        result, f_orig, f_filtered, mask, gray = apply_frequency_filter(
            test_img, radius_slider.value, filter_type_dropdown.value
        )

        # Schnelle Visualisierung
        fig, axes = plt.subplots(1, 3, figsize=(16, 5))

        axes[0].imshow(gray, cmap='gray')
        axes[0].set_title('Original', fontsize=13, fontweight='bold')
        axes[0].axis('off')

        axes[1].imshow(mask, cmap='gray')
        axes[1].set_title(f'{filter_type_dropdown.value.upper()}-Maske (R={radius_slider.value})',
                         fontsize=13, fontweight='bold')
        axes[1].axis('off')

        axes[2].imshow(result, cmap='gray')
        axes[2].set_title('Gefiltertes Ergebnis', fontsize=13, fontweight='bold')
        axes[2].axis('off')

        plt.tight_layout()
        plt.show()

        # Interpretation
        if filter_type_dropdown.value == 'lowpass':
            if radius_slider.value < 30:
                print("\n💡 Sehr kleiner Radius → Nur sehr niedrige Frequenzen → Sehr verschwommen")
            elif radius_slider.value < 70:
                print("\n💡 Mittlerer Radius → Gute Balance → Moderate Unschärfe")
            else:
                print("\n💡 Großer Radius → Viele Frequenzen behalten → Minimaler Effekt")
        else:
            if radius_slider.value < 40:
                print("\n💡 Kleiner Radius → Mehr Frequenzen entfernt → Starker Kanteneffekt")
            elif radius_slider.value < 80:
                print("\n💡 Mittlerer Radius → Moderater Effekt")
            else:
                print("\n💡 Großer Radius → Weniger entfernt → Schwacher Effekt")

filter_type_dropdown.observe(update_filter, names='value')
radius_slider.observe(update_filter, names='value')

print("🎮 Interaktiver Filter-Explorer")
print("   Probiere verschiedene Kombinationen und beobachte die Effekte!\n")
display(filter_type_dropdown, radius_slider, output_filter)
update_filter(None)

---

# 🎯 Teil 5: Fortgeschrittene Konzepte & Anwendungen

## 5.1 Warum ist die Fourier-Transformation wichtig?

### 📊 Vergleich Orts- vs. Frequenzbereich

| Aufgabe | Ortsbereich | Frequenzbereich |
|---------|------------|------------------|
| **Filterung** | Faltung (langsam bei großen Kerneln) | Multiplikation (schnell!) |
| **Verständnis** | Schwer Muster zu sehen | Klare Frequenzkomponenten |
| **Kompression** | Nicht intuitiv | Einfach (hohe Frequenzen verwerfen) |
| **Rauschentfernung** | Erfordert spezielle Filter | Einfache Frequenzauswahl |

### 💡 Das Faltungstheorem:

**Mathematische Magie:**
```
Faltung im Ortsbereich = Multiplikation im Frequenzbereich
```

**Warum ist das wichtig?**
- Große Kernel-Faltung: Langsam (O(n²))
- FFT + Multiplikation + IFFT: Schnell (O(n log n))
- Für große Filter ist Frequenzbereich VIEL schneller!

---

## 5.2 Reale Anwendung: JPEG-Kompression

**Wie JPEG tatsächlich funktioniert (vereinfacht):**

1. Teile Bild in 8×8 Blöcke
2. Wende **DCT** an (Diskrete Cosinus-Transformation - ähnlich FFT)
3. **Verwirf hohe Frequenzen** (kleine Koeffizienten)
4. Speichere nur wichtige Frequenzen

**Ergebnis:** 10× kleinere Datei, sieht aber fast identisch aus!

Simulieren wir das:

In [ ]:
def simulate_jpeg_compression(image, quality_levels=[10, 30, 60, 100]):
    """
    JPEG-ähnliche Kompression mittels Frequenzfilterung simulieren
    """
    if len(image.shape) == 3:
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    else:
        gray = image

    fig, axes = plt.subplots(1, len(quality_levels) + 1, figsize=(20, 4))

    # Original
    axes[0].imshow(gray, cmap='gray')
    axes[0].set_title('Original\n(100% Qualität)', fontsize=11, fontweight='bold')
    axes[0].axis('off')

    # Komprimierte Versionen
    for i, quality in enumerate(quality_levels, 1):
        # Verwende Radius als Proxy für Qualität
        # Niedrige Qualität = kleiner Radius = mehr Kompression
        compressed, _, _, _, _ = apply_frequency_filter(gray, quality, 'lowpass')

        axes[i].imshow(compressed, cmap='gray')

        if quality < 30:
            quality_label = "Niedrig"
            emoji = "🔴"
        elif quality < 70:
            quality_label = "Mittel"
            emoji = "🟡"
        else:
            quality_label = "Hoch"
            emoji = "🟢"

        axes[i].set_title(f'{emoji} {quality_label}e Qualität\n(Radius={quality})',
                         fontsize=11, fontweight='bold')
        axes[i].axis('off')

    plt.suptitle('JPEG-Kompressions-Simulation (via Frequenzfilterung)',
                fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print("\n💰 Kompressions-Trade-off:")
    print("   📦 Niedrige Qualität: Stark komprimiert (kleine Datei), merkbarer Qualitätsverlust")
    print("   ⚖️  Mittlere Qualität: Gute Balance (moderate Dateigröße), akzeptable Qualität")
    print("   🎯 Hohe Qualität: Weniger komprimiert (größere Datei), exzellente Qualität")
    print("\n   → JPEG verwirft hohe Frequenzen, die Menschen kaum wahrnehmen!")

# Simulation ausführen
print("📸 Simuliere JPEG-Kompression...\n")
simulate_jpeg_compression(data.camera())

## 5.3 Periodische Muster erkennen

**Anwendungsfall:** Wiederkehrende Muster in Bildern finden (Stoffe, Bildschirme, Gitter)

FFT macht periodische Muster offensichtlich - sie erscheinen als helle Punkte!

In [ ]:
# Bild mit periodischem Rauschen erstellen
def add_periodic_noise(image, frequency=20, amplitude=50):
    """
    Periodisches Rauschmuster hinzufügen (simuliert Bildschirmstörungen)
    """
    if len(image.shape) == 3:
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY).astype(float)
    else:
        gray = image.astype(float)

    # Periodisches Rauschen erstellen
    rows, cols = gray.shape
    x = np.arange(cols)
    y = np.arange(rows)
    X, Y = np.meshgrid(x, y)

    noise = amplitude * np.sin(2 * np.pi * frequency * X / cols)
    noisy = np.clip(gray + noise, 0, 255).astype(np.uint8)

    return noisy

# Verrauschtes Bild erstellen
clean = data.camera()
noisy = add_periodic_noise(clean, frequency=25, amplitude=70)

print("🔊 Bild mit periodischem Rauschen (wie Bildschirmstörungen)\n")

# Vergleich zeigen
fig, axes = plt.subplots(2, 2, figsize=(14, 14))

# Sauberes Bild
axes[0, 0].imshow(clean, cmap='gray')
axes[0, 0].set_title('Sauberes Bild', fontsize=13, fontweight='bold')
axes[0, 0].axis('off')

# Sauberes Spektrum
f_clean, _ = compute_fft(clean)
mag_clean = np.log(np.abs(f_clean) + 1)
axes[0, 1].imshow(mag_clean, cmap='hot')
axes[0, 1].set_title('Sauberes Spektrum', fontsize=13, fontweight='bold')
axes[0, 1].axis('off')

# Verrauschtes Bild
axes[1, 0].imshow(noisy, cmap='gray')
axes[1, 0].set_title('Verrauschtes Bild (Periodisches Muster)', fontsize=13, fontweight='bold')
axes[1, 0].axis('off')

# Verrauschtes Spektrum
f_noisy, _ = compute_fft(noisy)
mag_noisy = np.log(np.abs(f_noisy) + 1)
axes[1, 1].imshow(mag_noisy, cmap='hot')
axes[1, 1].set_title('Verrauschtes Spektrum (Siehst du die PUNKTE?)', fontsize=13, fontweight='bold')
axes[1, 1].axis('off')

# Rausch-Peaks markieren
center_y, center_x = mag_noisy.shape[0] // 2, mag_noisy.shape[1] // 2
#axes[1, 1].plot([center_x - 50, center_x + 50], [center_y, center_y], 'c--', linewidth=2)
#axes[1, 1].text(center_x + 60, center_y, 'Rausch-\nPeaks!', color='cyan', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n💡 Kernaussage:")
print("   ✓ Periodische Muster erscheinen als HELLE PUNKTE im FFT-Spektrum")
print("   ✓ Kann Frequenz und Richtung des Musters identifizieren")
print("   ✓ Kann die Punkte entfernen, um periodisches Rauschen zu eliminieren!")
print("\n   → FFT macht unsichtbare Muster sichtbar!")

---

# 🎓 Teil 6: Wichtigste Erkenntnisse & Zusammenfassung

## 📝 Das große Ganze

### Was wir gelernt haben:

1. **Räumliche Frequenzen sind intuitiv**
   - Niedrige Frequenz = glatte Bereiche, graduelle Änderungen
   - Hohe Frequenz = Kanten, Details, Texturen
   - Echte Bilder = Mischung ALLER Frequenzen

2. **FFT offenbart verborgene Struktur**
   - Transformiert Bild um Frequenzinhalt zu zeigen
   - Zentrum = niedrige Frequenzen, Ränder = hohe Frequenzen
   - Helligkeit = wie viel von jeder Frequenz

3. **Frequenzbereich ist mächtig**
   - Filterung ist einfache Multiplikation (vs. komplexe Faltung)
   - Kann spezifische Frequenzbereiche gezielt ansprechen
   - Macht Muster offensichtlich

4. **Echte Anwendungen überall**
   - JPEG-Kompression (hohe Frequenzen verwerfen)
   - Bildverbesserung (spezifische Frequenzen filtern)
   - Mustererkennung (periodische Strukturen finden)

---

## 🧠 Intuitions-Checkliste

**Können Sie diese Fragen beantworten ohne nachzuschaen?**

- Was bedeutet "niedrige Frequenz" in einem Bild?
- Was erscheint im Zentrum eines FFT-Spektrums?
- Warum hat eine scharfe Kante hohe Frequenzen?
- Was passiert wenn Sie hohe Frequenzen entfernen?
- Wie nutzt JPEG Frequenzinformationen?
- Warum im Frequenzbereich filtern statt im Ortsbereich?

---